In [15]:
import sys, importlib
MODULE_DIR = "/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/script"
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

import tommo_ld_tools
importlib.reload(tommo_ld_tools)

<module 'tommo_ld_tools' from '/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/script/tommo_ld_tools.py'>

In [16]:
import pickle

merge_dict_path = "/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/02.tommo_ld_compare/merged_tommo_focus_ld.pkl"

with open(merge_dict_path, "rb") as f:
    merged_tommo_focus_ld = pickle.load(f)


In [17]:
merged_tommo_focus_ld['chr3:154069965:A:G']['control']

,#CHROM_A,POS_A,ID_A,CHROM_B,POS_B,ID_B,TOMMO_R2,UNPHASED_R2,NOTE
0,3,154069965,chr3:154069965:A:G,3,154141824,chr3:154141824:G:A,0.898397,0.779034,NaN
1,3,154069965,chr3:154069965:A:G,3,154141825,chr3:154141825:A:T,0.898397,0.779031,NaN
2,3,154069965,chr3:154069965:A:G,3,154239812,chr3:154239812:G:A,0.515559,0.423829,NaN
3,3,154069965,chr3:154069965:A:G,3,154324772,chr3:154324772:G:T,0.510705,0.424611,NaN
4,3,154069965,chr3:154069965:A:G,3,154353612,chr3:154353612:C:A,0.387885,0.285357,NaN
5,3,154069965,chr3:154069965:A:G,3,154366576,chr3:154366576:C:T,0.507634,0.408650,NaN
6,3,154069965,chr3:154069965:A:G,3,154494156,chr3:154494156:G:A,0.363220,0.298184,NaN
7,3,154069965,chr3:154069965:A:G,3,154496495,chr3:154496495:G:A,0.363830,0.301282,NaN
8,3,154069965,chr3:154069965:A:G,3,154521262,chr3:154521262:C:T,0.332926,0.262158,NaN
9,3,154069965,chr3:154069965:A:G,3,154554258,chr3:154554258:C:G,0.364490,0.296129,NaN


In [18]:
# 获取control数据
control_data = merged_tommo_focus_ld['chr3:154069965:A:G']['control']

# 过滤掉UNPHASED_R2为NaN的行，并按UNPHASED_R2降序排序
filtered_data = control_data.dropna(subset=['UNPHASED_R2']).sort_values('UNPHASED_R2', ascending=False)

# 选取UNPHASED_R2最大的两个变体
top_2_variants = filtered_data.head(2)

print("Top 2 variants by UNPHASED_R2:")
print(top_2_variants[['ID_B', 'UNPHASED_R2']])

# 提取ID_B值
id_b_values = top_2_variants['ID_B'].values

# 保存到focus_loci.tsv文件（不包含列名）
import pandas as pd
output_df = pd.DataFrame(id_b_values)
output_df.to_csv('focus_loci.tsv', sep='\t', header=False, index=False)

print(f"\nSaved {len(id_b_values)} ID_B values to focus_loci.tsv:")
for id_b in id_b_values:
    print(id_b)

Top 2 variants by UNPHASED_R2:
                 ID_B  UNPHASED_R2
0  chr3:154141824:G:A     0.779034
1  chr3:154141825:A:T     0.779031

Saved 2 ID_B values to focus_loci.tsv:
chr3:154141824:G:A
chr3:154141825:A:T


In [19]:
import os

# 获取focus_loci.tsv的完整路径
focus_loci_path = os.path.abspath('focus_loci.tsv')
print(f"focus_loci.tsv的完整路径: {focus_loci_path}")

# 验证文件是否存在
if os.path.exists(focus_loci_path):
    print("文件存在")
else:
    print("文件不存在")

focus_loci.tsv的完整路径: /LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/03.specific_tommo_ld_compare/focus_loci.tsv
文件存在


In [20]:
tommo_ld_dir = "/LARGE0/gr10478/b37974/Pulmonary_Hypertension/ToMMo_60KJPN/co-occurrence"

In [21]:
from tommo_ld_tools import extract_ld_from_tommo_for_focus_loci 

tommo_ld_dict_path, tommo_ld_log = extract_ld_from_tommo_for_focus_loci(focus_loci_path, tommo_ld_dir)

tommo_ld_log

,ID,CHR,POS,Match_Found,Num_Matches
0,chr3:154141824:G:A,chr3,154141824,True,29
1,chr3:154141825:A:T,chr3,154141825,True,29


In [22]:
wgs_bed_prefix = "/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/wgs/19.tommo_panel_filter/cteph_agp3k.lowfreq_common"
case_prefix = "PHOM"
plink2_path = "/home/b/b37974/plink2_alpha6/plink2"
tommo_ld_dict = pd.read_pickle(tommo_ld_dict_path)

In [23]:
from tommo_ld_tools import compute_ld_between_focus_and_tommo_linked_variants

focus_ld_dict_path = compute_ld_between_focus_and_tommo_linked_variants(
    tommo_dict=tommo_ld_dict,
    bed_prefix=wgs_bed_prefix,
    case_prefix=case_prefix,
    plink2_path=plink2_path,
    threads=6
)

PLINK v2.0.0-a.6.20LM 64-bit Intel (7 Jul 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/03.specific_tommo_ld_compare/tmp/case_data.log.
Options in effect:
  --bfile /LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/wgs/19.tommo_panel_filter/cteph_agp3k.lowfreq_common
  --keep /LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/03.specific_tommo_ld_compare/tmp/case.keep
  --make-bed
  --out /LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/03.specific_tommo_ld_compare/tmp/case_data
  --threads 6

Start time: Thu Sep 18 12:29:43 2025
515039 MiB RAM detected, ~425138 available; reserving 257519 MiB for main
workspace.
Using up to 6 compute threads.
3017 samples (1934 females, 1083 males; 3017 founders) loaded from
/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/wgs/19.tommo

In [ ]:
from tommo_ld_tools import merge_tommo_and_focus_ld_dict

merged_dict_path = merge_tommo_and_focus_ld_dict(tommo_ld_dict_path, focus_ld_dict_path, select='control')

In [ ]:
from tommo_ld_tools import plot_ld_comparison_from_merged_dict

plot_ld_comparison_from_merged_dict(merged_dict_path, select='control')

'/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/03.specific_tommo_ld_compare/tommo_vs_focus_ld_scatter.pdf'